<div style="text-align: center;"><img src="https://www.databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" width="600"></div>


# Lecture — Type of Model Monitoring

## Overview

Once an ML solution reaches production, the work is not done — the world keeps changing, and a model that was accurate yesterday can quietly degrade today. This lecture explains **what** to monitor and **why**, the different **types of drift** that erode model quality, how those drifts show up in real-world scenarios, and the **triggers** that tell you it is time to retrain.

This lecture covers 4 sections that build on each other:

- **A. Why Monitor a Model**. The four dimensions of model monitoring — **Business Requirements**, **Model Performance**, **Resource Utilization**, and **Traceability**.
- **B. The Four Types of Drift**. **Data drift**, **concept drift**, **model quality drift**, and **bias drift** — what each one is, how it is detected, and how it looks when visualized.
- **C. Drift Scenarios in the Real World**. Concrete company examples that show data, concept, and bias drift as they actually appear in production.
- **D. Model Retraining Triggers**. The events — scheduled, data, code, configuration, and monitoring/alerts — that initiate a retraining job.

## Learning Objectives

By the end of this lecture, you will be able to:

1. Describe the four dimensions that model monitoring is responsible for.
2. Distinguish data drift, concept drift, model quality drift, and bias drift, and name the metrics used to detect them.
3. Recognize how each type of drift appears in a real-world business scenario.
4. Identify the triggers that should initiate model retraining.

## A. Why Monitor a Model

### A1. Four Dimensions of Model Monitoring

Model monitoring is not a single check — it spans **four distinct dimensions**, each protecting a different aspect of a production ML solution. Together they ensure the model keeps delivering value, stays accurate, runs efficiently, and remains auditable.

<div style="display: flex; gap: 16px; flex-wrap: wrap; margin: 16px 0; max-width: 1000px;">
  <div style="flex: 1; min-width: 300px; border: 1px solid #e0e0e0; border-top: 4px solid #FF3621; border-radius: 8px; padding: 16px 20px; background: #FFFFFF;">
    <div style="font-size: 18px; font-weight: 700; color: #FF3621; margin-bottom: 10px;">Business Requirements</div>
    <ul style="margin: 0; padding-left: 20px; font-size: 16px; line-height: 1.7; color: #333;">
      <li>Ensuring the ML solution <strong>aligns with and fulfills specific business objectives</strong></li>
      <li>Regular assessments to ensure <strong>continued relevance</strong> to evolving business needs</li>
    </ul>
  </div>
  <div style="flex: 1; min-width: 300px; border: 1px solid #e0e0e0; border-top: 4px solid #FF3621; border-radius: 8px; padding: 16px 20px; background: #FFFFFF;">
    <div style="font-size: 18px; font-weight: 700; color: #FF3621; margin-bottom: 10px;">Model Performance</div>
    <ul style="margin: 0; padding-left: 20px; font-size: 16px; line-height: 1.7; color: #333;">
      <li>Tracking the <strong>accuracy and efficiency</strong> of the model over time</li>
      <li>Detecting and addressing types of <strong>drift or degradation</strong></li>
    </ul>
  </div>
</div>
<div style="display: flex; gap: 16px; flex-wrap: wrap; margin: 16px 0; max-width: 1000px;">
  <div style="flex: 1; min-width: 300px; border: 1px solid #e0e0e0; border-top: 4px solid #FF3621; border-radius: 8px; padding: 16px 20px; background: #FFFFFF;">
    <div style="font-size: 18px; font-weight: 700; color: #FF3621; margin-bottom: 10px;">Resource Utilization</div>
    <ul style="margin: 0; padding-left: 20px; font-size: 16px; line-height: 1.7; color: #333;">
      <li>Ensuring <strong>efficient resource utilization</strong> within the ML infrastructure</li>
      <li><strong>Compliance</strong> with Service Level Agreements (<strong>SLAs</strong>) for system performance and availability</li>
    </ul>
  </div>
  <div style="flex: 1; min-width: 300px; border: 1px solid #e0e0e0; border-top: 4px solid #FF3621; border-radius: 8px; padding: 16px 20px; background: #FFFFFF;">
    <div style="font-size: 18px; font-weight: 700; color: #FF3621; margin-bottom: 10px;">Traceability</div>
    <ul style="margin: 0; padding-left: 20px; font-size: 16px; line-height: 1.7; color: #333;">
      <li>Facilitating <strong>audit trails</strong> for troubleshooting, regulatory compliance, and model improvement</li>
      <li><strong>Tracking data lineage</strong> to understand the origin, movement, and transformation of data</li>
    </ul>
  </div>
</div>

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout ul { margin: 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.info    { --accent:#1B5162; --tint:#F8F9FC; }
.caution { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
.danger  { --accent:#FF5F46; --tint:rgba(255,95,70,0.10); }
.success { --accent:#00A972; --tint:rgba(0,169,114,0.10); }
.ds-callout-title { font-size: 14pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
</style>

<div class="ds-callout caution">
  <div class="ds-eyebrow">Key Takeaway</div>
  <div class="ds-callout-title">Accuracy alone is not enough — monitoring must span all four dimensions.</div>
  <ul>
    <li>A model can score perfectly on accuracy yet still fail the business — by drifting away from the objective it was built for, breaching an SLA, or leaving no audit trail.</li>
    <li>Monitoring across all four dimensions keeps the solution <strong>relevant, accurate, efficient, and accountable</strong> throughout its production life.</li>
  </ul>
</div>
</div>

## B. The Four Types of Drift

### B1. The Four Types of Drift

**Drift** is the gradual mismatch between the world a model was trained on and the world it now operates in. There are four types to watch for — each with a different cause and a different set of detection metrics.

<div style="display: flex; gap: 16px; flex-wrap: wrap; margin: 16px 0; max-width: 1000px;">
  <div style="flex: 1; min-width: 300px; border: 1px solid #e0e0e0; border-top: 4px solid #FF3621; border-radius: 8px; padding: 16px 20px; background: #FFFFFF;">
    <div style="font-size: 18px; font-weight: 700; color: #FF3621; margin-bottom: 10px;">Data Drift</div>
    <ul style="margin: 0; padding-left: 20px; font-size: 16px; line-height: 1.7; color: #333;">
      <li>Occurs when the <strong>statistical properties</strong> of the input data <strong>change over time</strong></li>
      <li>Can impact the model's quality by <strong>introducing inconsistencies</strong> in the data patterns</li>
    </ul>
  </div>
  <div style="flex: 1; min-width: 300px; border: 1px solid #e0e0e0; border-top: 4px solid #FF3621; border-radius: 8px; padding: 16px 20px; background: #FFFFFF;">
    <div style="font-size: 18px; font-weight: 700; color: #FF3621; margin-bottom: 10px;">Concept Drift</div>
    <ul style="margin: 0; padding-left: 20px; font-size: 16px; line-height: 1.7; color: #333;">
      <li>Happens when the <strong>relationship</strong> between input features and the <strong>target variable changes</strong></li>
      <li>Forces models to adapt to <strong>new patterns</strong> to stay relevant</li>
    </ul>
  </div>
</div>
<div style="display: flex; gap: 16px; flex-wrap: wrap; margin: 16px 0; max-width: 1000px;">
  <div style="flex: 1; min-width: 300px; border: 1px solid #e0e0e0; border-top: 4px solid #FF3621; border-radius: 8px; padding: 16px 20px; background: #FFFFFF;">
    <div style="font-size: 18px; font-weight: 700; color: #FF3621; margin-bottom: 10px;">Model Quality Drift</div>
    <ul style="margin: 0; padding-left: 20px; font-size: 16px; line-height: 1.7; color: #333;">
      <li>Reflects a decrease in the model's <strong>predictive performance</strong> over time</li>
      <li>Can be <strong>detected through worsening metrics</strong> like accuracy, precision, recall, or F1 score</li>
    </ul>
  </div>
  <div style="flex: 1; min-width: 300px; border: 1px solid #e0e0e0; border-top: 4px solid #FF3621; border-radius: 8px; padding: 16px 20px; background: #FFFFFF;">
    <div style="font-size: 18px; font-weight: 700; color: #FF3621; margin-bottom: 10px;">Bias Drift</div>
    <ul style="margin: 0; padding-left: 20px; font-size: 16px; line-height: 1.7; color: #333;">
      <li>Involves shifts in model outcomes that could lead to <strong>unfair treatment</strong> of certain groups</li>
      <li>Monitoring for bias drift is <strong>crucial to maintain fairness and ethical standards</strong> in model predictions</li>
    </ul>
  </div>
</div>

<div style="border-left: 4px solid #4299E0; background: rgba(66,153,224,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#2272B4; margin-bottom:6px; font-size:15pt;">Info</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">The right metric depends on the problem type. For <strong>regression problems</strong>, watch <strong>Mean Squared Error (MSE)</strong> and <strong>R-squared</strong>. For <strong>classification problems</strong>, watch <strong>Accuracy, Precision, and Recall</strong>. A steady worsening of these metrics is the clearest signal that the model's quality is drifting.</div>
</div>

### B2. Illustrating Data & Model Drift

**Data drift** shows up as a change in the *shape* of an input distribution between training and production. **Model drift** shows up as a *degrading confusion matrix* — correct predictions on the diagonal give way to errors off it.

<div style="display: flex; gap: 24px; flex-wrap: wrap; margin: 16px 0; max-width: 900px;">
  <div style="flex: 1; min-width: 400px;">
    <div style="font-size: 16px; font-weight: 700; color: #1B3139; margin-bottom: 8px; text-align: center;">Data Drift</div>
    <svg width="100%" viewBox="0 0 400 440" role="img" style="font-family: sans-serif;">
      <title>Data drift — training vs production distribution of customer age</title>
      <!-- Training distribution (symmetric bell) - aligned with top confusion matrix -->
      <text x="250" y="18" text-anchor="middle" font-size="13" fill="#1B3139">Training data distribution</text>
      <rect x="100" y="30" width="300" height="150" fill="#F9F7F4" stroke="#B0BEC5"/>
      <path d="M110 178 C 160 178, 200 38, 250 35 C 300 35, 340 178, 390 178" fill="none" stroke="#FFAB00" stroke-width="2.5"/>
      <text x="250" y="198" text-anchor="middle" font-size="11" fill="#607D8B">Customer age</text>
      <text x="85" y="105" text-anchor="middle" font-size="10" fill="#607D8B" transform="rotate(-90 85 105)">Number of customers</text>
      <!-- Production distribution (skewed) - aligned with bottom confusion matrix -->
      <text x="250" y="235" text-anchor="middle" font-size="13" fill="#1B3139">Production data distribution</text>
      <rect x="100" y="248" width="300" height="150" fill="#F9F7F4" stroke="#B0BEC5"/>
      <path d="M110 396 C 210 394, 260 255, 300 253 C 340 255, 375 370, 390 396" fill="none" stroke="#FF3621" stroke-width="2.5"/>
      <text x="250" y="416" text-anchor="middle" font-size="11" fill="#607D8B">Customer age</text>
      <text x="85" y="323" text-anchor="middle" font-size="10" fill="#607D8B" transform="rotate(-90 85 323)">Number of customers</text>
    </svg>
    <p style="font-size: 13px; color: #607D8B; margin: 4px 0 0 0;">The input distribution shifts and skews — the model still expects the old shape.</p>
  </div>
  <div style="flex: 1; min-width: 400px;">
    <div style="font-size: 16px; font-weight: 700; color: #1B3139; margin-bottom: 8px; text-align: center;">Model Drift</div>
    <svg width="100%" viewBox="0 0 400 440" role="img" style="font-family: sans-serif;">
      <title>Model drift — confusion matrix right after deployment vs after a period of time</title>
      <!-- TOP MATRIX: Right after deployment -->
      <text x="220" y="18" text-anchor="middle" font-size="12" fill="#1B3139" font-weight="bold">Right After Deployment</text>
      <!-- Left axis labels -->
      <text x="95" y="105" text-anchor="middle" font-size="11" fill="#607D8B" transform="rotate(-90 95 105)">Predicted Label</text>
      <text x="125" y="68" text-anchor="middle" font-size="10" fill="#607D8B" transform="rotate(-90 125 68)">Positive</text>
      <text x="125" y="143" text-anchor="middle" font-size="10" fill="#607D8B" transform="rotate(-90 125 143)">Negative</text>
      <!-- Grid cells -->
      <rect x="145" y="30" width="75" height="75" fill="#B71C1C"/><text x="183" y="75" text-anchor="middle" font-size="20" font-weight="bold" fill="#FFF">80</text>
      <rect x="220" y="30" width="75" height="75" fill="#5C6BC0"/><text x="258" y="75" text-anchor="middle" font-size="20" font-weight="bold" fill="#FFF">5</text>
      <rect x="145" y="105" width="75" height="75" fill="#5C6BC0"/><text x="183" y="150" text-anchor="middle" font-size="20" font-weight="bold" fill="#FFF">10</text>
      <rect x="220" y="105" width="75" height="75" fill="#B71C1C"/><text x="258" y="150" text-anchor="middle" font-size="20" font-weight="bold" fill="#FFF">85</text>
      <!-- Bottom axis labels -->
      <text x="183" y="196" text-anchor="middle" font-size="10" fill="#607D8B">Positive</text>
      <text x="258" y="196" text-anchor="middle" font-size="10" fill="#607D8B">Negative</text>
      <text x="220" y="210" text-anchor="middle" font-size="11" fill="#607D8B">True Label</text>
      <!-- BOTTOM MATRIX: After a period of time -->
      <text x="220" y="235" text-anchor="middle" font-size="12" fill="#1B3139" font-weight="bold">After a Period of Time</text>
      <!-- Left axis labels -->
      <text x="95" y="323" text-anchor="middle" font-size="11" fill="#607D8B" transform="rotate(-90 95 323)">Predicted Label</text>
      <text x="125" y="286" text-anchor="middle" font-size="10" fill="#607D8B" transform="rotate(-90 125 286)">Positive</text>
      <text x="125" y="361" text-anchor="middle" font-size="10" fill="#607D8B" transform="rotate(-90 125 361)">Negative</text>
      <!-- Grid cells -->
      <rect x="145" y="248" width="75" height="75" fill="#D7CCC8"/><text x="183" y="293" text-anchor="middle" font-size="20" font-weight="bold" fill="#1B3139">50</text>
      <rect x="220" y="248" width="75" height="75" fill="#90CAF9"/><text x="258" y="293" text-anchor="middle" font-size="20" font-weight="bold" fill="#1B3139">35</text>
      <rect x="145" y="323" width="75" height="75" fill="#5C6BC0"/><text x="183" y="368" text-anchor="middle" font-size="20" font-weight="bold" fill="#FFF">10</text>
      <rect x="220" y="323" width="75" height="75" fill="#B71C1C"/><text x="258" y="368" text-anchor="middle" font-size="20" font-weight="bold" fill="#FFF">85</text>
      <!-- Bottom axis labels -->
      <text x="183" y="414" text-anchor="middle" font-size="10" fill="#607D8B">Positive</text>
      <text x="258" y="414" text-anchor="middle" font-size="10" fill="#607D8B">Negative</text>
      <text x="220" y="428" text-anchor="middle" font-size="11" fill="#607D8B">True Label</text>
    </svg>
    <p style="font-size: 13px; color: #607D8B; margin: 4px 0 0 0;">True positives fall from 80 to 50 while false positives climb from 5 to 35 — accuracy is drifting.</p>
  </div>
</div>

<div style="border-left: 4px solid #4299E0; background: rgba(66,153,224,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#2272B4; margin-bottom:6px; font-size:15pt;">Info</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">A model trained on a <strong>symmetric distribution</strong> will have performance issues once the production data becomes <strong>skewed</strong> — that is data drift. When the confusion matrix decays even though the inputs look similar, the <em>relationship</em> has changed — a sign of concept or model quality drift.</div>
</div>

### B3. Illustrating Concept & Bias Drift

**Concept drift** changes the *function* that maps inputs to outputs — the same feature predicts a different outcome depending on context. **Bias drift** appears as prediction accuracy diverging *across groups* as their distribution shifts.

<div style="display: flex; gap: 24px; flex-wrap: wrap; margin: 16px 0; max-width: 1000px;">
  <div style="flex: 1; min-width: 320px;">
    <div style="font-size: 15px; font-weight: 700; color: #1B3139; margin-bottom: 8px; text-align: center;">Concept Drift</div>
    <svg width="100%" viewBox="0 0 380 190" role="img" style="font-family: sans-serif;">
      <title>Concept drift — likelihood to buy iced tea vs temperature, summer vs winter</title>
      <!-- Summer: strong S-curve -->
      <text x="95" y="16" text-anchor="middle" font-size="11" fill="#1B3139" font-weight="bold">Summer</text>
      <line x1="40" y1="30" x2="40" y2="150" stroke="#1B3139" stroke-width="1.5"/>
      <line x1="40" y1="150" x2="180" y2="150" stroke="#1B3139" stroke-width="1.5"/>
      <path d="M45 145 C 90 145, 100 40, 175 38" fill="none" stroke="#FF3621" stroke-width="2.5"/>
      <text x="110" y="168" text-anchor="middle" font-size="9" fill="#607D8B">Temperature</text>
      <text x="30" y="90" text-anchor="middle" font-size="8" fill="#607D8B" transform="rotate(-90 30 90)">Likelihood to buy iced tea</text>
      <!-- Winter: flat line -->
      <text x="290" y="16" text-anchor="middle" font-size="11" fill="#1B3139" font-weight="bold">Winter</text>
      <line x1="235" y1="30" x2="235" y2="150" stroke="#1B3139" stroke-width="1.5"/>
      <line x1="235" y1="150" x2="375" y2="150" stroke="#1B3139" stroke-width="1.5"/>
      <path d="M240 140 C 290 138, 330 135, 372 133" fill="none" stroke="#FF3621" stroke-width="2.5"/>
      <text x="305" y="168" text-anchor="middle" font-size="9" fill="#607D8B">Temperature</text>
    </svg>
    <p style="font-size: 13px; color: #607D8B; margin: 4px 0 0 0;">Temperature strongly predicts iced-tea sales in summer, but barely in winter — the concept has drifted.</p>
  </div>
  <div style="flex: 1; min-width: 320px;">
    <div style="font-size: 15px; font-weight: 700; color: #1B3139; margin-bottom: 8px; text-align: center;">Bias Drift</div>
    <svg width="100%" viewBox="0 0 380 190" role="img" style="font-family: sans-serif;">
      <title>Bias drift — change in prediction accuracy across groups</title>
      <line x1="40" y1="30" x2="40" y2="150" stroke="#1B3139" stroke-width="1.5"/>
      <line x1="40" y1="95" x2="360" y2="95" stroke="#1B3139" stroke-width="1"/>
      <!-- Bars: some positive (above axis), some negative (below) -->
      <rect x="60" y="70" width="34" height="25" fill="#FF3621"/>
      <rect x="110" y="95" width="34" height="30" fill="#FF3621"/>
      <rect x="160" y="82" width="34" height="13" fill="#FF3621"/>
      <rect x="210" y="95" width="34" height="45" fill="#FF3621"/>
      <rect x="260" y="60" width="34" height="35" fill="#FF3621"/>
      <rect x="310" y="95" width="34" height="18" fill="#FF3621"/>
      <text x="77" y="164" text-anchor="middle" font-size="8" fill="#607D8B">Grp A</text>
      <text x="127" y="164" text-anchor="middle" font-size="8" fill="#607D8B">Grp B</text>
      <text x="177" y="164" text-anchor="middle" font-size="8" fill="#607D8B">Grp C</text>
      <text x="227" y="164" text-anchor="middle" font-size="8" fill="#607D8B">Grp D</text>
      <text x="277" y="164" text-anchor="middle" font-size="8" fill="#607D8B">Grp E</text>
      <text x="327" y="164" text-anchor="middle" font-size="8" fill="#607D8B">Grp F</text>
      <text x="30" y="60" text-anchor="middle" font-size="8" fill="#607D8B" transform="rotate(-90 30 60)">Change in accuracy</text>
    </svg>
    <p style="font-size: 13px; color: #607D8B; margin: 4px 0 0 0;">Prediction accuracy changes unevenly across groups — some improve, others degrade, revealing bias drift.</p>
  </div>
</div>

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#1B5162; margin-bottom:6px; font-size:15pt;">Note</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">Bias drift occurs when there is a <strong>shift in the group distribution</strong> in production relative to the training data. The <strong>emergence of a new group</strong> not represented in training — or a feature that has <strong>limited data for a small group of customers</strong> — can degrade model performance for exactly the people it under-represents.</div>
</div>

## C. Drift Scenarios in the Real World

### C1. Data Drift Scenario — A New Product Line

**Event: Introduction of a New Product Line.** A company introduces a line of **eco-friendly products**, heavily promoted through social media and email campaigns. This new line attracts a **different demographic** compared to the existing customer base — and the model's assumptions no longer hold.

<br>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-title { font-size: 16pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card-text { font-size: 12pt; color: #5E7077; line-height: 1.55; margin: 0; }
.ds-card ul, .ds-card ol { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
.tc { text-align: center; }
</style>

<div class="ds-row">
  <div class="ds-card blue"><div class="ds-card-title">Changes in Data</div><ul><li>Demographic shift</li><li>Browsing behavior</li><li>Historical sales data</li><li>Promotional activities</li></ul></div>
  <div class="ds-card coral"><div class="ds-card-title">Impact of Data Drift</div><ul><li><strong>Prediction Accuracy:</strong> training data no longer represents current customer behavior and product offerings</li><li><strong>Sales Forecasting:</strong> eco-friendly sales are underestimated while other products are overestimated</li></ul></div>
  <div class="ds-card green"><div class="ds-card-title">Addressing Data Drift</div><ul><li>Collect new data</li><li>Retrain the model</li><li>Monitor continuously</li></ul></div>
</div>
</div>

<div style="border-left: 4px solid #4299E0; background: rgba(66,153,224,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#2272B4; margin-bottom:6px; font-size:15pt;">Info</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">New eco-conscious customers spend more time reading product descriptions and reviews, historical sales data does not include the new line, and <strong>social media becomes a bigger driver of sales than email</strong>. Notice this scenario also <em>overlaps with concept drift</em> — the meaning of the same features has shifted, not just their distribution.</div>
</div>

### C2. Concept & Bias Drift Scenarios

Two more scenarios show how **concept drift** and **bias drift** surface in production ML systems.

<br>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-headcard { background: #F9F7F4; border-radius: 10px; overflow: hidden; box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.blue{--accent:#2272B4;} .green{--accent:#00A972;}
.ds-headband { background: var(--accent); color: #fff; padding: 14px 18px; font-size: 16pt; font-weight: 700; text-align: center; }
.ds-headbody { padding: 18px 20px; font-size: 12pt; color: #0b2026; line-height: 1.55; }
.ds-headbody ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-headbody li { margin-bottom: 10px; }
.ds-headbody li:last-child { margin-bottom: 0; }
.ds-note { margin-top: 14px; background: #F8F9FC; border-left: 4px solid var(--accent); border-radius: 6px; padding: 10px 12px; font-size: 14pt; color: #0b2026; }
</style>

<div class="ds-row">
  <div class="ds-headcard blue">
    <div class="ds-headband">Concept Drift — AgriSmart</div>
    <div class="ds-headbody">
      <div style="font-size: 15px; color: #607D8B; margin-bottom: 8px;">An agricultural technology firm</div>
      <ul>
        <li><strong>Initial training data:</strong> crop-yield data using factors like soil quality, moisture, nutrient levels, and pest presence</li>
        <li><strong>Concept drift:</strong> an invasive species changes the <em>relationship</em> between an existing feature — pest presence — and crop yield, so the same inputs now map to different outcomes (P(Y|X) has changed)</li>
        <li><strong>A separate change:</strong> a new farming technique introduces a sixth feature, "genetic modifications" — adding a new input is a schema/feature change rather than concept drift itself, though it also calls for retraining</li>
        <li><strong>Result:</strong> the model underestimates these effects on yields, forecasts become less accurate, affecting crop-management and resource-allocation decisions</li>
      </ul>
      <div class="ds-note"><strong>Guidance:</strong> Monitor for new features and changing relationships; retrain when production data diverges from training assumptions.</div>
    </div>
  </div>
  <div class="ds-headcard green">
    <div class="ds-headband">Bias Drift — GlobalMusic</div>
    <div class="ds-headbody">
      <div style="font-size: 15px; color: #607D8B; margin-bottom: 8px;">An international streaming service</div>
      <ul>
        <li><strong>Initial training data:</strong> user interaction data across various music genres</li>
        <li><strong>Production change:</strong> introduction of new regional markets with distinct musical preferences alters user demographics and genre popularity</li>
        <li><strong>Result:</strong> the recommendation engine under-recommends regional music to new demographic groups, reducing engagement and subscription renewals in new markets</li>
      </ul>
      <div class="ds-note"><strong>Guidance:</strong> Monitor group-level performance; retrain and adjust for new populations to maintain fairness and relevance.</div>
    </div>
  </div>
</div>
</div>

<br>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout ul { margin: 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.info    { --accent:#1B5162; --tint:#F8F9FC; }
.caution { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
.danger  { --accent:#FF5F46; --tint:rgba(255,95,70,0.10); }
.success { --accent:#00A972; --tint:rgba(0,169,114,0.10); }
.ds-callout-title { font-size: 14pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
</style>

<div class="ds-callout caution">
  <div class="ds-eyebrow">Key Takeaway</div>
  <div class="ds-callout-title">The model didn't change — the world it models did.</div>
  <ul>
    <li>Concept drift changed <em>which relationships matter</em> (AgriSmart).</li>
    <li>Bias drift changed <em>who the population is</em> (GlobalMusic).</li>
    <li>Only monitoring makes these invisible shifts visible in time to act.</li>
  </ul>
</div>
</div>

## D. Model Retraining Triggers

### D1. What Triggers a Retraining Job

When monitoring reveals that a model has drifted, the response is usually **retraining**. Databricks recommends **starting with scheduled, periodic retraining and moving to triggered retraining when needed**. Five categories of trigger initiate a retraining job.

<br>
<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif;">
<style>
.rsd-row { display: block; white-space: nowrap; font-size: 0; width: 100%; box-sizing: border-box; }
.rsd-box { display: inline-block; width: 18.4%; margin-right: 2%; min-height: 100px; background: #F9F7F4; border-top: 8px solid transparent; border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent; border-radius: 8px; padding: 14px 10px; text-align: center; box-sizing: border-box; cursor: pointer; user-select: none; vertical-align: top; transition: transform 0.12s; box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.rsd-box:last-child { margin-right: 0; }
.rsd-box:hover { transform: translateY(-2px); }
.rsd-box.active { background: #fff; border-left-color: var(--pc); border-right-color: var(--pc); border-bottom-color: var(--pc); }
.rsd-title { display: block; font-size: 12pt; font-weight: 700; color: #0b2026; line-height: 1.3; white-space: normal; pointer-events: none; }
.rsd-num { display: block; font-size: 20pt; font-weight: 800; margin-bottom: 4px; pointer-events: none; }
.rsd-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top: 0; }
.rsd-detail-wrap.open { max-height: 800px; opacity: 1; margin-top: 12px; }
.rsd-detail-card { background: #F9F7F4; border-radius: 10px; padding: 20px; box-sizing: border-box; border-top: 7px solid #ccc; box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.rsd-block { border-left: 4px solid; border-radius: 0 6px 6px 0; padding: 10px 12px; margin-bottom: 12px; }
</style>

<div style="background: #F8F9FC; border: 3px solid #1B5162; border-radius: 10px; padding: 24px; box-sizing: border-box;">
  <div class="rsd-row">
    <span class="rsd-box" data-id="0" onclick="rsdD1Select(0)" style="--pc:#4299E0; border-top-color:#4299E0;"><span class="rsd-num" style="color:#4299E0;">1</span><span class="rsd-title">Scheduled Retraining</span></span><span class="rsd-box" data-id="1" onclick="rsdD1Select(1)" style="--pc:#FF5F46; border-top-color:#FF5F46;"><span class="rsd-num" style="color:#FF5F46;">2</span><span class="rsd-title">Data Changes</span></span><span class="rsd-box" data-id="2" onclick="rsdD1Select(2)" style="--pc:#FFAB00; border-top-color:#FFAB00;"><span class="rsd-num" style="color:#FFAB00;">3</span><span class="rsd-title">Model Code Changes</span></span><span class="rsd-box" data-id="3" onclick="rsdD1Select(3)" style="--pc:#00A972; border-top-color:#00A972;"><span class="rsd-num" style="color:#00A972;">4</span><span class="rsd-title">Configuration Changes</span></span><span class="rsd-box" data-id="4" onclick="rsdD1Select(4)" style="--pc:#98102A; border-top-color:#98102A;"><span class="rsd-num" style="color:#98102A;">5</span><span class="rsd-title">Monitoring &amp; Alerts</span></span>
  </div>
  <div class="rsd-detail-wrap" id="rsd-d1-wrap">
    <div class="rsd-detail-card" id="rsd-d1-card"></div>
  </div>
</div>
</div>

<script>
var RSDD1 = [
  { title: "Scheduled Retraining", color: "#4299E0", tintBg: "rgba(66,153,224,0.10)",
    points: ["Periodic retraining on a fixed cadence.", "Recommended starting point for production ML.", "Keeps model up-to-date with recent data."],
    example: "Retrain a recommendation system every month to include the latest user behavior." },
  { title: "Data Changes", color: "#FF5F46", tintBg: "rgba(255,95,70,0.10)",
    points: ["Explicit changes in the data can trigger retraining.", "Automated retraining when data drift is detected.", "Ensures model reflects current data patterns."],
    example: "A holiday sales spike requires the sales-prediction model to be retrained." },
  { title: "Model Code Changes", color: "#FFAB00", tintBg: "rgba(255,171,0,0.12)",
    points: ["Retraining triggered by changes in the model code.", "Often due to concept drift or new features.", "Maintains model relevance and accuracy."],
    example: "Introducing a new feature requires retraining to incorporate it effectively." },
  { title: "Model Configuration Changes", color: "#00A972", tintBg: "rgba(0,169,114,0.10)",
    points: ["Alterations in model configuration can trigger retraining.", "Includes hyperparameter tuning and architecture changes.", "Optimizes model performance."],
    example: "Tuning learning rate or batch size to improve performance." },
  { title: "Monitoring & Alerts", color: "#98102A", tintBg: "rgba(152,16,42,0.10)",
    points: ["Jobs monitor for data and model drift.", "SQL dashboards display status and send alerts.", "Automated retraining when drift or degradation is detected."],
    example: "Retrain a fraud-detection model when a rise in false positives is detected (model quality drift)." }
];
var rsdD1Current = null;
function rsdD1Select(id) {
  var wrap = document.getElementById('rsd-d1-wrap');
  var card = document.getElementById('rsd-d1-card');
  var p = RSDD1[id];
  document.querySelectorAll('.rsd-box').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id && rsdD1Current !== id);
  });
  if (rsdD1Current === id) { wrap.classList.remove('open'); rsdD1Current = null; return; }
  rsdD1Current = id;
  var pointsHtml = p.points.map(function(c){ return '<li style="margin-bottom:6px;">' + c + '</li>'; }).join('');
  card.style.borderTopColor = p.color;
  card.innerHTML =
    '<div style="font-size:20pt;font-weight:700;margin-bottom:12px;color:#0b2026;">' + p.title + '</div>'
    + '<div class="rsd-block" style="background:#F8F9FC;border-color:' + p.color + ';">'
    + '<div style="font-weight:700;font-size:15pt;margin-bottom:4px;">Description</div>'
    + '<ul style="font-size:14pt;margin:0;padding-left:16px;line-height:1.55;">' + pointsHtml + '</ul></div>'
    + '<div class="rsd-block" style="background:' + p.tintBg + ';border-color:' + p.color + ';margin-bottom:0;">'
    + '<div style="font-weight:700;font-size:15pt;margin-bottom:4px;">Example</div>'
    + '<div style="font-size:14pt;color:#0b2026;line-height:1.55;">' + p.example + '</div></div>';
  wrap.classList.add('open');
}
</script>

<div style="border-left: 4px solid #00A972; background: rgba(0,169,114,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#00A972; margin-bottom:6px; font-size:15pt;">Success</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">Begin with simple scheduled retraining to establish a reliable baseline, then graduate to <strong>drift-triggered</strong> retraining — where monitoring and SQL alerts automatically kick off a Lakeflow Job — as your MLOps practice matures.</div>
</div>

## Conclusion

In this lecture, you covered the types of model monitoring and the drift they detect:

1. **Why Monitor a Model** — Monitoring spans four dimensions: **Business Requirements** (staying aligned to objectives), **Model Performance** (accuracy and drift), **Resource Utilization** (efficiency and SLAs), and **Traceability** (audit trails and data lineage).
2. **The Four Types of Drift** — **Data drift** (input distribution changes), **concept drift** (input-to-target relationship changes), **model quality drift** (predictive metrics worsen — MSE/R² for regression, accuracy/precision/recall for classification), and **bias drift** (group distribution shifts affecting fairness).
3. **Drift Scenarios in the Real World** — A new eco-friendly product line drives data drift; AgriSmart's new farming feature drives concept drift; GlobalMusic's new regional markets drive bias drift. In each, the world changed, not the model.
4. **Model Retraining Triggers** — Retraining is initiated by **scheduled** cadence, **data** changes, **model code** changes, **configuration** changes, and **monitoring/alerts** — starting scheduled and moving to triggered as you mature.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>